# Hollywood by the Numbers: Clustering Analysis
- Alex Arce: aarce
- Lunden Mandigo: lundenm
- Tyrone Pettygrue: tpetty

## Preprocessing Data

In [11]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA


In [3]:
movies = pd.read_csv('movies.csv')

# Drop unecessary columns
movies = movies.drop(columns=['originalTitle', 'main_genre', 'rating_category'])

# Combine the 'title' column and 'year' column to drop duplicate rows but keep movies with the same title
movies['title'] = movies['title'] + ' (' + movies['year'].astype(str) + ')'
movies = movies.drop_duplicates(subset=['title'])

# Make the title column the index
movies = movies.set_index('title')

# Split the 'genres' and 'productionCountries' columns from strings into lists
movies['genres'] = movies['genres'].str.split(', ')
movies['productionCountries'] = movies['productionCountries'].str.split(', ')

# Replace NaN values in the new 'genres' and 'productionCountries' columns with empty lists
movies[['genres', 'productionCountries']] = movies[['genres', 'productionCountries']].fillna(value={})

# Drop all values that are not list objects from the 'productionCountries' column
movies['productionCountries'] = movies['productionCountries'].apply(lambda x: x if isinstance(x, list) else ([x] if pd.notna(x) else []))



In [4]:
movies.head()


,isAdult,runtimeMinutes,genres,IMDBavgRating,numVotes,rank,worldwideGross,domesticGross,domestic%,foreignGross,foreign%,year,originalLang,productionCountries
title,,,,,,,,,,,,,,
Kate & Leopold (2001),0,118,"[Comedy,Fantasy,Romance]",6.4,91304,56,76019048.0,47121859.0,62.0,28897189.0,38.0,2001,en,[United States of America]
The Sorcerer's Apprentice (2010),0,86,"[Adventure,Family,Fantasy]",4.2,757,34,215283742.0,63150991.0,29.3,152132751.0,70.7,2010,en,[United States of America]
Fantastic Four (2005),0,106,"[Action,Adventure,Fantasy]",5.7,353227,11,333535934.0,154696080.0,46.4,178839854.0,53.6,2005,en,"[Germany, United States of America]"
Fantastic Four (2015),0,106,"[Action,Adventure,Fantasy]",5.7,353227,44,167882881.0,56117548.0,33.4,111765333.0,66.6,2015,en,"[United Kingdom, Germany, United States of Ame..."
Frida (2002),0,123,"[Biography,Drama,Romance]",7.3,97408,78,56298474.0,25885000.0,46.0,30413474.0,54.0,2002,en,"[Canada, Mexico, United States of America]"


In [5]:
movies.productionCountries.value_counts().sort_values(ascending=True).head(10)

productionCountries
[Canada, France, Norway, United Kingdom, United States of America]    1
[Italy, South Africa, United Kingdom, United States of America]       1
[Canada, France, Germany, Switzerland]                                1
[Belgium, France, Germany, Spain, United States of America]           1
[Germany, United States of America, Brazil]                           1
[Italy, Spain]                                                        1
[France, United States of America, Germany]                           1
[United Kingdom, United States of America, Croatia]                   1
[Germany, Canada, United States of America]                           1
[United States of America, Canada, Germany]                           1
Name: count, dtype: int64

In [9]:
# Onehot encode categorical columns with a MultiLabelBinarizer
mlb = MultiLabelBinarizer()
encoded_genres = mlb.fit_transform(movies['genres'])
encoded_genres_df = pd.DataFrame(encoded_genres, columns=mlb.classes_)
encoded_genres_df = encoded_genres_df.fillna(0)
encoded_production_countries = mlb.fit_transform(movies['productionCountries'])
encoded_production_countries_df = pd.DataFrame(encoded_production_countries, columns=mlb.classes_)
encoded_production_countries_df = encoded_production_countries_df.fillna(0)



# concat the encoded columns with the original dataframe
processed_movies = pd.concat([movies, encoded_genres_df, encoded_production_countries_df], axis=1)

# drop the original columns
processed_movies = processed_movies.drop(columns=['genres'])
processed_movies = processed_movies.drop(columns=['productionCountries'])
# processed_movies = processed_movies.drop(columns=['originalLang'])



In [10]:
Yid = processed_movies.index

In [12]:
categorical_cols = ['originalLang']
numerical_cols = ['runtimeMinutes', 'IMDBavgRating', 'numVotes', 'rank', 'worldwideGross', 'domesticGross', 'domestic%', 'foreignGross', 'foreign%', "year"]


num_pipeline = Pipeline([
    ('impute',SimpleImputer(strategy='median')), 
    ('scale',StandardScaler())
    ])

preprocessing_pipeline = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', OneHotEncoder(drop='if_binary'), categorical_cols)
    ])

In [13]:
# apply the pipeline to my data
scaled_X = preprocessing_pipeline.fit_transform(processed_movies)

In [14]:
# apply PCA to the scaled data
model = PCA(n_components=7)
X_pca = model.fit_transform(scaled_X)

# create a dataframe from the PCA data
pca_df = pd.DataFrame(X_pca,index=Yid, columns=[f'PC{i}' for i in range(1, 8)])
pca_df.head()
model.explained_variance_ratio_


array([0.37315892, 0.214096  , 0.11744063, 0.0840028 , 0.06632624,
       0.05315587, 0.03850452])

In [15]:
pca_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7
Kate & Leopold (2001),0.896061,2.279608,0.887411,-1.246331,0.845740,-0.782039,0.156796
The Sorcerer's Apprentice (2010),0.937418,0.225093,-3.434508,-0.949285,0.888324,-0.876352,0.719884
Fantastic Four (2005),3.927429,0.948364,-0.943775,-1.442382,0.598895,-0.506675,0.970831
Fantastic Four (2015),1.845119,-0.214946,-0.802197,0.164656,0.122854,-0.055949,1.915743
Frida (2002),0.516794,1.079201,1.872714,-1.309653,0.254311,-0.430248,-0.062360


In [16]:
# write the data to a csv
pca_df.to_csv('preprocessed_movie_data.csv')
